# StateLM Fig. 2 on the real deployment stream

This notebook reproduces the online FSM-growth experiment with **real trials only** (plus 36 labelled augmented requests) and the **already fine-tuned Qwen3-4B SLM** from `robot_slm_runs/qwen3_robot_slm_20260916_032017/robot_slm_bundle`. No retraining.

What happens, in order:

1. Mount Drive, unpack the bundled code and data (embedded below, hash-checked), install the pinned SLM runtime.
2. Controller tests (deterministic test double; no GPU).
3. Build FSM_0 seed data from the 41 success trials and the deployment stream from the 93 recovery trials (37 participants, participant order) with the augmented rows interleaved: 222 turns.
4. Run the trained SLM once on every stream payload and cache the raw decisions on Drive (about 220 calls; roughly 30 min on an A100, 1 to 2 h on a T4; resumable after a disconnect).
5. Replay the online routing from the cache (MiniLM embedding, fixed thresholds 0.78 / 0.82 / 0.85, admission after 2 distinct observations), save logs, metrics, snapshot.
6. Draw the three-panel figure and a sensitivity table.
7. **Standalone chat** (last cell): reloads everything it needs from Drive and lets you type user turns and injected robot lines.

Runtime: **T4 GPU** (free tier) works; A100 is faster. Runtime > Change runtime type > GPU. A TPU runtime cannot run the SLM. The paths in cell 1 match your Drive (shortcut `robot_slm_runs` in My Drive, outputs in `My Drive/statelm_fig2_real/run_v1`).

In [ ]:
#@title 1. Mount Drive, set paths, unpack the bundled package
import base64, hashlib, io, json, os, shutil, sys, zipfile
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

BUNDLE_DIR = "/content/drive/MyDrive/robot_slm_runs/qwen3_robot_slm_20260916_032017/robot_slm_bundle" #@param {type:"string"}
WORK_DIR = "/content/drive/MyDrive/statelm_fig2_real/run_v1" #@param {type:"string"}
EXPECTED_ADAPTER_SHA256 = "049a71886e7b413ba90e69f8e2b8ab545c851a2bf2974275ddee40042ce7c4a6"

PACKAGE_DIR = Path('/content/statelm_realstream_v1')
PACKAGE_SHA256 = "c23fd760766b747ba0c4215206c9e4523e6ac7d61897078a5f0fd109131ca669"
PACKAGE_BASE64 = """
...
"""
payload = base64.b64decode("".join(PACKAGE_BASE64.split()))
assert hashlib.sha256(payload).hexdigest() == PACKAGE_SHA256, "embedded package is corrupted"
if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    archive.extractall('/content')
work = Path(WORK_DIR)
work.mkdir(parents=True, exist_ok=True)
(work / 'statelm_realstream_v1.zip').write_bytes(payload)   # copy for the standalone chat cell
bundle = Path(BUNDLE_DIR)
if not (bundle / 'runtime_config.json').exists():
    # Fall back to a search: the shared folder may be mounted under another name.
    import glob
    found = sorted(glob.glob('/content/drive/MyDrive/*/qwen3_robot_slm_20260916_032017/robot_slm_bundle/runtime_config.json')
                   + glob.glob('/content/drive/MyDrive/*/*/qwen3_robot_slm_20260916_032017/robot_slm_bundle/runtime_config.json')
                   + glob.glob('/content/drive/MyDrive/qwen3_robot_slm_20260916_032017/robot_slm_bundle/runtime_config.json')
                   + glob.glob('/content/drive/MyDrive/robot_slm_bundle/runtime_config.json'))
    if found:
        bundle = Path(found[0]).parent
        print('bundle located at', bundle)
    else:
        raise SystemExit(
            'SLM bundle not found. In Google Drive open the shared folder robot_slm_runs '
            'click its name at the top, '
            'Organise > Add shortcut > My Drive, then run this cell again.')
os.environ['BUNDLE_DIR'] = str(bundle)
os.environ['WORK_DIR'] = str(work)
os.environ['EXPECTED_ADAPTER_SHA256'] = EXPECTED_ADAPTER_SHA256
os.chdir(PACKAGE_DIR)
if str(PACKAGE_DIR) not in sys.path:
    sys.path.insert(0, str(PACKAGE_DIR))
print('package unpacked to', PACKAGE_DIR)
print('outputs will be written to', work)
print('SLM bundle:', bundle)

In [ ]:
#@title 2. Check the GPU, install the pinned SLM runtime and the embedding library
import importlib.metadata as metadata, subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise SystemExit('No GPU in this runtime. Runtime > Change runtime type > T4 GPU (free tier) or A100, then run again.')
print('GPU:', gpu.stdout.strip())
!python -m pip -q install -r "$BUNDLE_DIR/requirements.txt"
!python -m pip -q install "sentence-transformers>=5,<6" "pandas>=2.0" "matplotlib>=3.8" ipywidgets
# Colab can leave newer Transformers/PEFT files behind; pin the trained-runtime pair.
!python -m pip -q install --force-reinstall --no-deps transformers==4.57.1 peft==0.17.1 accelerate==1.10.1
assert metadata.version('transformers') == '4.57.1'
assert metadata.version('peft') == '0.17.1'
assert metadata.version('accelerate') == '1.10.1'
print('Runtime ready: transformers 4.57.1, peft 0.17.1, accelerate 1.10.1')

In [ ]:
#@title 3. Controller tests (deterministic test double, no GPU, no SLM)
!python -m unittest discover -s tests -v

In [ ]:
#@title 4. Build FSM_0 seed data and the real deployment stream
!python scripts/build_stream.py --out "$WORK_DIR/data"

import json, os, pandas as pd
from pathlib import Path
work = Path(os.environ['WORK_DIR'])
stream = [json.loads(l) for l in (work / 'data' / 'stream.jsonl').read_text().splitlines() if l.strip()]
rows = []
for r in stream[:14]:
    p = r['input']
    rows.append({'t': r['position'], 'kind': r['kind'], 'origin': r['origin'].split('_')[0],
                 'participant': r['participant_id'], 'setting': r['setting'],
                 'user': p['conversation'][-1]['text'][:60],
                 'proposal': (p.get('proposed_robot_answer') or '')[:45],
                 'failure': '+'.join(r['failure_types'])})
display(pd.DataFrame(rows))

In [ ]:
#@title 5. Run the trained SLM once on every stream payload (cached on Drive, resumable)
!python scripts/teacher_pass.py --bundle "$BUNDLE_DIR" --stream "$WORK_DIR/data/stream.jsonl" --cache "$WORK_DIR/slm_cache.jsonl" --expected-adapter-sha256 "$EXPECTED_ADAPTER_SHA256"

In [ ]:
#@title 6. Replay the online routing from the cache (main configuration)
!python scripts/replay_stream.py --seed "$WORK_DIR/data/seed.jsonl" --stream "$WORK_DIR/data/stream.jsonl" --slm-cache "$WORK_DIR/slm_cache.jsonl" --out "$WORK_DIR/replay_main" --embedder sentence-transformer --embedding-model sentence-transformers/all-MiniLM-L6-v2 --device cuda --merge-similarity 0.78 --routing-similarity 0.82 --admission-threshold 0.85 --minimum-observations 2

In [ ]:
#@title 7. Figure 2 (three panels) from the main replay
!python scripts/make_figure.py --replay "$WORK_DIR/replay_main" --out "$WORK_DIR/fig2_real_stream.png"
import json, os
from pathlib import Path
from IPython.display import Image, display
display(Image(filename=f"{os.environ['WORK_DIR']}/fig2_real_stream.png", width=900))

m = json.loads((Path(os.environ['WORK_DIR']) / 'replay_main' / 'metrics.json').read_text())
print(json.dumps(m['summary'], indent=2))

In [ ]:
#@title 8. Sensitivity of the FSM share to thresholds and embedding (from the same cache; no SLM calls)
import itertools, json, os, subprocess, pandas as pd
from pathlib import Path
work = Path(os.environ['WORK_DIR'])
configs = []
for tau_r, min_obs in itertools.product([0.78, 0.82, 0.86, 0.90], [2, 3, 5]):
    configs.append(dict(model='sentence-transformers/all-MiniLM-L6-v2', tau_z=min(0.78, tau_r - 0.04), tau_r=tau_r, min_obs=min_obs, seed=True))
for tau_r in [0.85, 0.90, 0.94]:
    configs.append(dict(model='BAAI/bge-small-en-v1.5', tau_z=tau_r - 0.02, tau_r=tau_r, min_obs=2, seed=True))
configs.append(dict(model='sentence-transformers/all-MiniLM-L6-v2', tau_z=0.78, tau_r=0.82, min_obs=2, seed=False))
rows = []
for i, c in enumerate(configs):
    out = work / 'sensitivity' / f'cfg_{i:02d}'
    cmd = ['python', 'scripts/replay_stream.py', '--seed', str(work / 'data' / 'seed.jsonl'),
           '--stream', str(work / 'data' / 'stream.jsonl'), '--slm-cache', str(work / 'slm_cache.jsonl'),
           '--out', str(out), '--embedder', 'sentence-transformer', '--embedding-model', c['model'],
           '--device', 'cuda', '--merge-similarity', str(c['tau_z']), '--routing-similarity', str(c['tau_r']),
           '--admission-threshold', '0.85', '--minimum-observations', str(c['min_obs']), '--quiet']
    if not c['seed']:
        cmd.append('--no-seed')
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        print('config', i, 'failed:', proc.stderr[-800:])
        continue
    s = json.loads((out / 'metrics.json').read_text())['summary']
    rows.append({'embedding': c['model'].split('/')[-1], 'tau_z': c['tau_z'], 'tau_r': c['tau_r'],
                 'min_obs': c['min_obs'], 'FSM_0 seeded': c['seed'],
                 'FSM share overall': round(100 * s['fsm_share_overall'], 1),
                 'FSM share last 25': round(100 * s['fsm_share_last_window'], 1),
                 'FSM share last third': round(100 * s['fsm_share_by_third'][2], 1),
                 'FSM=SLM agreement': None if s['fsm_served_exact_agreement_with_checked_slm'] is None else round(100 * s['fsm_served_exact_agreement_with_checked_slm'], 1),
                 'action vs reference': round(100 * s['served_action_matches_reference'], 1),
                 'wrong item served': s['wrong_item_served']})
table = pd.DataFrame(rows)
table.to_csv(work / 'sensitivity_table.csv', index=False)
display(table)

## Reading the results

* **Panel A** is the quantity the paper's Fig. 2 claims: the share of turns served by the FSM without an SLM call, over the deployment turn index. Nothing here is assumed; every SLM decision is the trained model's own cached output.
* **Panel B** shows how many transitions exist (active = executable, candidate = stored after one observation but not yet admitted). FSM_0's seed patterns are the dotted line.
* **Panel C** is the safety side: FSM-served decisions that equal the SLM's checked decision, served actions that equal the reference action, and correct failure types on review turns. `wrong item served` in the footer must be 0.
* `metrics.json` also splits the FSM share by turn kind (request / review / augmented) and by SLM fold (`slm_test` = participants the SLM never saw).
* The augmented rows are the only inputs that were not recorded; `augmented_action_ok_by_category` reports them separately, and the figure can be regenerated without them by filtering `turn_log.jsonl` on `origin`.

## Standalone chat with the learned StateLM

The next cell is self-sufficient: run it alone in a fresh runtime (after Colab disconnected) and it mounts Drive, unpacks the package copy saved in `WORK_DIR`, loads the learned FSM snapshot and the trained SLM, and opens a chat box.

* Type a request, e.g. `Where are the gloves?`
* Inject a failed robot line with the prefix `robot:`, e.g. `robot: The blue tape is in drawer 1.` The system shows what the pre-send review decides and what it would have said instead; the injected line is then treated as spoken.
* Follow with a correction, e.g. `I said gloves.`
* Commands: `/reset`, `/setting lab`, `/setting hospital`, `/learn on`, `/learn off` (learning is off by default so the demo does not alter the learned cache).

Every reply names the requested item and the drawer read from the live inventory; a cached transition cannot store or invent a drawer.

In [ ]:
!pip -q uninstall -y torchao

In [ ]:
#@title 9. Standalone chat (self-sufficient: run this cell alone after a disconnect)
import importlib, importlib.util, json, os, shutil, sys, zipfile
from pathlib import Path

BUNDLE_DIR = "/content/drive/MyDrive/robot_slm_runs/qwen3_robot_slm_20260916_032017/robot_slm_bundle" #@param {type:"string"}
WORK_DIR = "/content/drive/MyDrive/statelm_fig2_real/run_v1" #@param {type:"string"}
SNAPSHOT_PATH = "" #@param {type:"string"}
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2" #@param {type:"string"}
EXPECTED_ADAPTER_SHA256 = "049a71886e7b413ba90e69f8e2b8ab545c851a2bf2974275ddee40042ce7c4a6"

# Drive and package
if not Path('/content/drive/MyDrive').exists():
    from google.colab import drive
    drive.mount('/content/drive')
work = Path(WORK_DIR)
package_dir = Path('/content/statelm_realstream_v1')
if not (package_dir / 'statelm_realstream' / 'router.py').exists():
    package_zip = work / 'statelm_realstream_v1.zip'
    assert package_zip.exists(), f'{package_zip} not found; run cell 1 once to create it'
    with zipfile.ZipFile(package_zip) as archive:
        archive.extractall('/content')
os.chdir(package_dir)
if str(package_dir) not in sys.path:
    sys.path.insert(0, str(package_dir))

# dependencies (installed only when missing)
def _have(module):
    return importlib.util.find_spec(module) is not None
if not (_have('peft') and _have('sentence_transformers') and _have('ipywidgets')):
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-r', f'{BUNDLE_DIR}/requirements.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'sentence-transformers>=5,<6', 'ipywidgets'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '--force-reinstall', '--no-deps',
                    'transformers==4.57.1', 'peft==0.17.1', 'accelerate==1.10.1'], check=True)

import ipywidgets as widgets
from IPython.display import display
from statelm_realstream import PaperAlignedRouter, SentenceTransformerEmbedder
from statelm_realstream.session import CombinedSession, format_record
from statelm_realstream.contract import validate_decision, decision_dict

# learned FSM snapshot and inventories
snapshot_path = Path(SNAPSHOT_PATH) if SNAPSHOT_PATH else work / 'replay_main' / 'snapshot.json'
assert snapshot_path.exists(), f'{snapshot_path} not found; run cells 4 to 6 first'
snapshot = json.loads(snapshot_path.read_text())
inventories = json.loads((package_dir / 'data' / 'inventory_by_setting.json').read_text())
print('FSM snapshot:', snapshot_path, '|', sum(p['status'] == 'active' for p in snapshot['patterns']), 'active patterns')

# trained SLM (loaded once per runtime)
import hashlib
bundle = Path(BUNDLE_DIR)
digest = hashlib.sha256()
with (bundle / 'adapter' / 'adapter_model.safetensors').open('rb') as handle:
    while block := handle.read(2 ** 20):
        digest.update(block)
assert digest.hexdigest() == EXPECTED_ADAPTER_SHA256, 'adapter does not match the verified checkpoint'
if 'SLM_ENGINE' not in globals():
    spec = importlib.util.spec_from_file_location('robot_slm_runtime', bundle / 'slm_runtime.py')
    Runtime = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = Runtime
    spec.loader.exec_module(Runtime)
    print('Loading the trained SLM; this can take several minutes...')
    import torch
    assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU or A100'
    small_gpu = torch.cuda.get_device_properties(0).total_memory < 12e9
    SLM_ENGINE = Runtime.load_runtime(bundle, quantize=small_gpu)   # fp16/bf16 on T4 or A100, 4-bit only on small GPUs
    print('SLM loaded.')

class LiveSLMTeacher:
    """The trained SLM; its raw JSON is validated by the router's task guard."""
    def __call__(self, payload):
        return SLM_ENGINE(payload)

import torch
embedder = SentenceTransformerEmbedder(EMBEDDING_MODEL, device='cuda' if torch.cuda.is_available() else 'cpu')
router = PaperAlignedRouter.from_snapshot(snapshot, LiveSLMTeacher(), inventories['lab'], embedder=embedder)
chat = CombinedSession(router, inventories, setting='lab', learn=False)

# widgets
box = widgets.Text(placeholder='Where are the gloves?   |   robot: The blue tape is in drawer 1.   |   /reset', description='You:', layout=widgets.Layout(width='80%'))
send = widgets.Button(description='Send', button_style='primary')
reset = widgets.Button(description='Reset conversation')
log = widgets.Output(layout={'border': '1px solid #bbb', 'padding': '8px'})

def handle(text):
    text = text.strip()
    if not text:
        return
    with log:
        try:
            if text == '/reset':
                chat.reset(); print('--- conversation reset ---'); return
            if text.startswith('/setting'):
                chat.set_setting(text.split()[-1]); print(f'--- setting = {chat.setting}, conversation reset ---'); return
            if text.startswith('/learn'):
                chat.learn = text.split()[-1].lower() == 'on'; print(f'--- learning {"on" if chat.learn else "off"} ---'); return
            # if text.lower().startswith('robot:'):
            #     print(format_record(chat.robot(text[6:])))
            # else:
            #     print(format_record(chat.user(text)))
            import re as _re
            parts = [p.strip() for p in _re.split(r'\||\n', text)
                     if p.strip()]
            for seg in parts:
                low = seg.lower()
                if low.startswith('robot:'):
                    print(format_record(chat.robot(seg[6:])))
                else:
                    if low.startswith('user:'):
                        seg = seg[5:].strip()
                    print(format_record(chat.user(seg)))
        except Exception as exc:
            print('Error:', type(exc).__name__, exc)
        print()

def on_send(_=None):
    value = box.value
    box.value = ''
    handle(value)

send.on_click(on_send)
reset.on_click(lambda _: handle('/reset'))
try:
    box.on_submit(on_send)          # ipywidgets 7 (Colab default)
except Exception:
    box.continuous_update = False   # ipywidgets 8: Enter commits the value
    box.observe(lambda change: on_send() if change['new'] else None, names='value')
display(widgets.HBox([box, send, reset]), log)

# built-in check of the wrong-item scenario
with log:
    print('Self-check: wrong item injected, then a correction')
    r1 = chat.user('Where are the gloves?'); print(format_record(r1))
    r2 = chat.robot('The blue tape is in drawer 1.'); print(format_record(r2))
    r3 = chat.user('I said gloves.'); print(format_record(r3))
    ok = (r2['review']['item'] == 'gloves' and 'drawer 4' in r2['what_statelm_would_have_said']
          and r3['decision']['resolved_item_id'] == 'gloves' and 'drawer 4' in r3['response']
          and 'tape' not in r3['response'])
    print('SELF-CHECK', 'PASS' if ok else 'FAIL', '(requested item and its drawer preserved)')
    chat.reset(); print('--- conversation reset; type your own turns above ---\n')